# Defect Formation Energy of a Boron Vacancy in h-BN

> **Fabian Bertoldo, Sajid Ali, Simone Manti & Kristian S. Thygesen**,
> "Quantum point defects in 2D materials - the QPOD database", npj Computational Materials, 2022.
> [DOI:10.1038/s41524-022-00730-w](https://doi.org/10.1038/s41524-022-00730-w)

Computes the neutral formation energy of the vacancy created in the
[structure notebook](defect_point_vacancy_boron_nitride.ipynb) and compares it with QPOD's
`v_B in BN (charge 0)` entry, using the generic
[Defect Formation Energy](../workflows/defect_formation_energy.ipynb) workflow.

$$E_f = E_{\text{defective}} - E_{\text{pristine}} - \Delta N_B\, \mu_B \quad [\text{eV}]$$

$\mu_B$ is the total energy of boron's standard state per atom (QPOD's convention, $q = 0$). For
V_B, $\Delta N_B = -1$ (one boron removed), so the last term adds $\mu_B$.

QPOD's value is for a relaxed cell (84 atoms pristine, 83 defective; 15.06 Å defect spacing); this
notebook's 48/47-atom cell is unrelaxed -- each effect is below 0.05 eV (measured with MACE).

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](defect_point_vacancy_boron_nitride.ipynb) first.
1. Set the parameters in cells 1.2-1.4 (or use the defaults), then "Run" > "Run All".
1. The default run submits up to four Quantum ESPRESSO jobs (pristine, B, N, defect) and waits for
   them, which can take hours; reference jobs are reused on a rerun.
1. The last line reports whether the recomputed E_f reproduces Bertoldo et al. (2022).

## Summary

Loads the materials, resolves the elemental references, configures one model and per-material
k-grids, creates and waits for the prerequisite and defect jobs, then compares the result with QPOD.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")


### 1.2. Material names

In [ ]:
# Names saved by defect_point_vacancy_boron_nitride.ipynb.
PRISTINE_NAME = "h-BN supercell"
DEFECTIVE_NAME = "B-vacancy h-BN"


### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

TOTAL_ENERGY_SEARCH_TERM = "total_energy.json"
DEFECT_WORKFLOW_SEARCH_TERM = "defect_formation_energy.json"
MY_WORKFLOW_NAME = "Defect Formation Energy"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = "cluster-001"
QUEUE_NAME = QueueName.OF
PPN = 40
TIME_LIMIT = "04:00:00"  # one hour is not enough for a spin-polarized SCF on this cell

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds


### 1.4. DFT model parameters

In [ ]:
FUNCTIONAL = "pbe"
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the platform's default PBE family for B and N
ECUTWFC = 40   # Ry, GBRV's recommended wavefunction cutoff
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff

# QPOD: 6 Å⁻¹ for relaxations, 12 for ground states; 6 used here for cost -- the neutral E_f
# does not need the denser grid.
KPOINT_DENSITY = 6
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry"


## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account

In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load the materials
### 3.1. Load from the uploads folder, and print provenance

In [ ]:
from collections import Counter
from mat3ra.notebooks_utils.material import load_material_from_folder

def formula(material):
    counts = Counter(material.basis.elements.values)
    return "".join(f"{element}{counts[element]}" for element in sorted(counts))

materials_by_name = {}
for name in (PRISTINE_NAME, DEFECTIVE_NAME):
    material = load_material_from_folder(FOLDER, name, verbose=False)
    if material is None:
        raise ValueError(f"No material named '{name}' in '{FOLDER}'. Run "
                         "defect_point_vacancy_boron_nitride.ipynb first, or correct the name above.")
    materials_by_name[name] = material

pristine, defective = materials_by_name[PRISTINE_NAME], materials_by_name[DEFECTIVE_NAME]
for name, material in materials_by_name.items():
    a, b = material.lattice.a, material.lattice.b
    print(f"{name}: {formula(material)}, {material.basis.number_of_atoms} atoms, "
          f"cell {a:.2f} x {b:.2f} Å, min in-plane vector {min(a, b):.2f} Å")


### 3.2. Resolve elemental reference materials

Elemental references are platform materials, not Standata entries: the workflow itself resolves
`{'tags': 'elemental', 'metadata.element': ...}` and then `total_energy` properties by that
material's `exabyteId`, which an uploaded copy never shares with the platform's own seed.

In [ ]:
from mat3ra.made.material import Material

elements = sorted(set(pristine.basis.elements.values) | set(defective.basis.elements.values))

elemental_materials_data = client.materials.list({"tags": "elemental", "metadata.element": {"$in": elements}})
elemental_materials = {}
for element in elements:
    matches = [m for m in elemental_materials_data if m.get("metadata", {}).get("element") == element]
    if not matches:
        raise ValueError(f"No platform elemental reference material tagged for {element}; "
                         "seed one (tags=['elemental'], metadata.element) first.")
    elemental_materials[element] = matches[0]
    atoms = Material.create(matches[0]).basis.number_of_atoms
    print(f"{element}: {matches[0]['name']} ({matches[0]['_id']}, owner {matches[0]['owner']['slug']}, "
          f"{atoms} atoms)")


### 3.3. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_pristine = get_or_create_material(client, pristine, ACCOUNT_ID)
saved_defective = get_or_create_material(client, defective, ACCOUNT_ID)


## 4. Configure the shared DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

cutoffs_context = PlanewaveCutoffsContextProvider(
    wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True).get_context_item_data()
print(f"Using application: {app.name}, model: {MODEL_TAG}")


### 4.2. k-grid per material

In [ ]:
import math

# made.Material objects, purely for lattice/basis access -- kept alongside the platform dicts
# in elemental_materials, which are used for ids and job creation.
material_objects = {PRISTINE_NAME: pristine, DEFECTIVE_NAME: defective,
                    **{element: Material.create(data) for element, data in elemental_materials.items()}}

def kgrid_for_density(material, periodic_dims=(0, 1, 2)):
    # |b_i| = 2*pi*reciprocal_vector_norms[i]; dims outside periodic_dims stay at 1 (vacuum).
    norms = material.lattice.reciprocal_vector_norms
    grid = [1, 1, 1]
    for dim in periodic_dims:
        grid[dim] = max(1, math.ceil(KPOINT_DENSITY * 2 * math.pi * norms[dim]))
    return grid

kgrid = {
    PRISTINE_NAME: kgrid_for_density(pristine, periodic_dims=(0, 1)),
    DEFECTIVE_NAME: kgrid_for_density(defective, periodic_dims=(0, 1)),
    **{element: kgrid_for_density(material_objects[element]) for element in elemental_materials},
}
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")


## 5. Configure compute
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")


## 6. Prerequisite Total Energy jobs

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_scf_kgrid
from mat3ra.notebooks_utils.job import create_job

total_energy_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    TOTAL_ENERGY_SEARCH_TERM
)

# Reuse is keyed on this exact workflow name, which carries the model, so a finished job at other
# cutoffs is not a hit -- the defect workflow resolves the pristine energy separately, by highest
# precision among this material's own qe: total_energy properties, and the results cell's
# consistency check catches it if that resolves to a job other than this one. Jobs on
# curators-owned materials (the elementals) are allowed.
prerequisite_materials = {PRISTINE_NAME: saved_pristine, **elemental_materials}
prerequisite_job_ids = {}
new_job_ids = []
for name, saved_material in prerequisite_materials.items():
    workflow_name = f"Total Energy {name} {MODEL_TAG}"
    existing = client.jobs.list(
        {"_material._id": saved_material["_id"], "owner._id": ACCOUNT_ID, "status": "finished",
         "workflow.name": workflow_name},
        {"sort": {"updatedAt": -1}, "limit": 1},
    )
    if existing:
        print(f"♻️  {name}: reusing existing Total Energy job {existing[0]['_id']}")
        prerequisite_job_ids[name] = existing[0]["_id"]
        continue
    workflow = Workflow.create(total_energy_workflow_config)
    workflow.name = workflow_name
    subworkflow = workflow.subworkflows[0]
    subworkflow.model = model
    unit = subworkflow.get_unit_by_name(name="pw_scf")
    unit.add_context(cutoffs_context)
    subworkflow.set_unit(unit)
    apply_scf_kgrid(workflow, kgrid[name], material=material_objects[name])

    job_response = create_job(
        api_client=client, materials=[saved_material], workflow=workflow, project_id=project_id,
        owner_id=ACCOUNT_ID, prefix=f"{workflow.name} {timestamp}", compute=compute.to_dict(),
    )
    prerequisite_job_ids[name] = job_response["_id"]
    new_job_ids.append(job_response["_id"])
    print(f"✅ {name}: created Total Energy job {job_response['_id']}")


In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

if new_job_ids:
    submit_jobs(client.jobs, new_job_ids)
    print(f"✅ Submitted {len(new_job_ids)} prerequisite job(s).")
    await wait_for_jobs_to_finish_async(client.jobs, new_job_ids, poll_interval=POLL_INTERVAL)

job_statuses = {name: client.jobs.get(job_id)["status"] for name, job_id in prerequisite_job_ids.items()}
unfinished = {name: status for name, status in job_statuses.items() if status != "finished"}
if unfinished:
    raise RuntimeError(f"Prerequisite job(s) not finished: {unfinished}")


## 7. Configure the Defect Formation Energy workflow

In [ ]:
from mat3ra.notebooks_utils.workflow import patch_workflow_qe_input
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

defect_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    DEFECT_WORKFLOW_SEARCH_TERM
)
defect_workflow = Workflow.create(defect_workflow_config)
defect_workflow.name = f"{MY_WORKFLOW_NAME} {MODEL_TAG}"

for subworkflow in defect_workflow.subworkflows:
    if subworkflow.name == "Compute Total Energy for Defective Material":
        subworkflow.model = model
        unit = subworkflow.get_unit_by_name(name="pw_scf")
        unit.add_context(cutoffs_context)
        subworkflow.set_unit(unit)
    elif subworkflow.name == "Resolve Total Energies for Elemental Materials":
        # References resolve from this account (falling back to curators), not the workflow's
        # own default 'public' (any owner, highest precision wins).
        source_unit = subworkflow.get_unit_by_name(name="assign-source-of-te-for-an-element")
        source_unit.value = "'my_account'"
        subworkflow.set_unit(source_unit)
apply_scf_kgrid(defect_workflow, kgrid[DEFECTIVE_NAME], material=defective)

# nspin/magnetization on the defective cell's SCF only -- species order is B, N, and the V_B
# moment sits on the N neighbours, so both are given an initial moment.
patch_workflow_qe_input(defect_workflow, {"system": {"nspin": 2, "starting_magnetization(1)": 0.5,
                                                     "starting_magnetization(2)": 0.5}},
                        unit_names=["pw_scf"])

visualize_workflow(defect_workflow)


## 8. Create and run the Defect Formation Energy job
### 8.1. Create the job (defective + pristine)

In [ ]:
defect_job_response = create_job(
    api_client=client, materials=[saved_defective, saved_pristine], workflow=defect_workflow,
    project_id=project_id, owner_id=ACCOUNT_ID, compute=compute.to_dict(),
    prefix=f"{MY_WORKFLOW_NAME} {formula(defective)} {timestamp}",
)
defect_job_id = defect_job_response["_id"]
print(f"✅ Defect Formation Energy job created: {defect_job_id}")


### 8.2. Submit and monitor the job

In [ ]:
client.jobs.submit(defect_job_id)
print(f"✅ Job {defect_job_id} submitted successfully!")
await wait_for_jobs_to_finish_async(client.jobs, [defect_job_id], poll_interval=POLL_INTERVAL)

status = client.jobs.get(defect_job_id)["status"]
if status != "finished":
    raise RuntimeError(f"Defect Formation Energy job {defect_job_id} did not finish (status: {status}).")


## 9. Retrieve the results
### 9.1. Defect formation energy

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

defect_energy_data = get_properties_for_job(client, defect_job_id, property_name="defect_formation_energy")
if not defect_energy_data:
    raise RuntimeError(f"Job {defect_job_id} produced no defect_formation_energy -- check it finished.")
visualize_properties(defect_energy_data, title="Defect Formation Energy")
e_formation = defect_energy_data[0]["value"]

def total_energy_for(job_id, label):
    data = get_properties_for_job(client, job_id, property_name="total_energy")
    if not data:
        raise RuntimeError(f"Job {job_id} ({label}) has no total_energy property.")
    return data[0]["value"]

# Consistency check: recompute E_f from the three total energies this notebook owns, and flag
# it if the workflow resolved a different reference (e.g. a curators' energy outranking ours).
e_def = total_energy_for(defect_job_id, "defective")
e_pris = total_energy_for(prerequisite_job_ids[PRISTINE_NAME], "pristine")
b_atoms = material_objects["B"].basis.number_of_atoms
e_b_per_atom = total_energy_for(prerequisite_job_ids["B"], "B reference") / b_atoms
e_formation_check = e_def - e_pris + e_b_per_atom
reference_matches = abs(e_formation - e_formation_check) <= 0.001
print(f"μ_B = {e_b_per_atom:.4f} eV/atom ({b_atoms} atoms)")
print(f"E_f (workflow): {e_formation:.3f} eV, E_f (recomputed): {e_formation_check:.3f} eV")
if not reference_matches:
    print("⚠️  The workflow resolved a different reference energy than this notebook's own jobs.")


### 9.2. Compare with QPOD

Only the neutral (q = 0) defect is compared: QPOD's charged states need a finite-size correction
this workflow does not compute. QPOD's 10.18 eV is itself PBE/PAW (GPAW) theory, not experiment.

In [ ]:
QPOD = {"standard_states": 10.18, "B_poor": 8.89}  # eV, QPOD v_B in BN (charge 0)

difference = e_formation - QPOD["standard_states"]
verdict = "yes" if abs(difference) <= 0.5 else "no"
note = "" if reference_matches else " -- reference mismatch, see warning above"
print(f"E_f (this notebook):        {e_formation:.3f} eV")
print(f"E_f (QPOD, standard states): {QPOD['standard_states']:.3f} eV")
print(f"E_f (QPOD, B-poor):          {QPOD['B_poor']:.3f} eV")
print(f"Difference from standard states: {difference:+.3f} eV")
print(f"Reproduces Bertoldo et al. (2022): {verdict}{note}")
